In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage("부산은 지금 몇시야?")])

AIMessage(content='지금은 **[현재 시간]** 입니다.\n\n(예시: 만약 지금이 서울 기준으로 오후 3시 30분이라면, "지금은 오후 3시 30분입니다.")\n\n부산은 서울과 같은 한국 표준시(KST)를 사용하므로, 서울 시간과 동일합니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c12f9-8139-7032-a684-dedd855d69f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 649, 'total_tokens': 657, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 581}})

In [10]:
from datetime import datetime
from zoneinfo import ZoneInfo

seoul_tz = ZoneInfo("Asia/Seoul")
print(seoul_tz)
print(type(seoul_tz))

Asia/Seoul
<class 'zoneinfo.ZoneInfo'>


In [13]:
newyork_tz = ZoneInfo("America/New_York")

print(datetime.now(tz=seoul_tz))
print(datetime.now(tz=newyork_tz))

2026-01-31 16:39:53.542028+09:00
2026-01-31 02:39:53.543036-05:00


In [15]:
from langchain_core.tools import tool

# 1. 타입 힌팅
# 2. doc-string
# 3. 데코레이터

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: Asia/Seoul) 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)

    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f"{timezone} ({location}) 현재 시각 {now}"

    print(location_and_local_time)

    return location_and_local_time

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[get_current_time],
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 도구를 사용할 수 있다."
)

In [20]:
res = agent.invoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]}
)

Asia/Seoul (부산) 현재 시각 2026-01-31 17:03:52


In [24]:
res["messages"][-1].content

'부산의 현재 시각은 2026년 01월 31일 17시 03분 52초 입니다.'